In [38]:
library(Seurat)
library(SingleCellExperiment)
library(magrittr)
library(Matrix.utils)
library(dplyr)
library(stringr)
library(tibble)
library(zellkonverter)

In [39]:
step <- function(msg) {
    cat(sprintf("\n[%s] %s\n", format(Sys.time(), "%H:%M:%S"), msg))
    flush.console()
}

In [ ]:
# ---- Acer & Amel ------------------------------------------------------------
# # Acer
# in_file <- "./matrix/MN_R_input/Acer4MN_8910.h5ad"
# label_col <- "leiden_res0.50"
# RDS_out_path <- "./matrix/MN_R_dataset/Acer_raw_8910.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Acer_markers_8910.tsv"
#! ↑ Acer Done
# # Amel
# in_file <- "./matrix/MN_R_input/Amel4MN_8910.h5ad"
# label_col <- "cell_type_cluster"
# RDS_out_path <- "./matrix/MN_R_dataset/Amel_raw_8910.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Amel_markers_8910.tsv"
# set.seed(0)
#! ↑ Amel Done
# ---- Acer & Hsal ------------------------------------------------------------
# Acer
# in_file <- "./matrix/MN_R_input/Acer4MN_8073.h5ad"
# label_col <- "leiden_res0.50"
# RDS_out_path <- "./matrix/MN_R_dataset/Acer_raw_8073.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Acer_markers_8073.tsv"
#! ↑ Acer Done
# Hsal
# in_file <- "./matrix/MN_R_input/Hsal4MN_8073.h5ad"
# label_col <- "cluster_name"
# RDS_out_path <- "./matrix/MN_R_dataset/Hsal_raw_8073.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Hsal_markers_8073.tsv"
#! ↑ Hsal Done
# ---- Acer & Dmel ------------------------------------------------------------
# Acer
# in_file <- "./matrix/MN_R_input/Acer4MN_6171.h5ad"
# label_col <- "leiden_res0.50"
# RDS_out_path <- "./matrix/MN_R_dataset/Acer_raw_6171.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Acer_markers_6171.tsv"
#! ↑ Acer Done
# Dmel
# in_file <- "./matrix/MN_R_input/Dmel4MN_6171_onlybrain.h5ad"
# label_col <- "afca_annotation"
# RDS_out_path <- "./matrix/MN_R_dataset/Dmel_raw_6171_onlybrain.RDS"
# marker_out_path <- "./metadata/MN_R_markers/Dmel_markers_6171_onlybrain.tsv"
#! ↑ Dmel Done

In [41]:
# ---- 1. input: h5ad -> SCE (replaces reference seurat.RDS / matrix.RDS+metadata)
step("1. read h5ad with counts as X and no layers")
sp_sce <- readH5AD(in_file, X_name = "counts", layers = FALSE)
# Amel <- readH5AD("./matrix/MN_R_input/Amel4MN_8910.h5ad", X_name = "counts", layers = FALSE)
cat("SCE dim:", paste(dim(sp_sce), collapse = " x "), "\n")
cat("assays:", paste(assayNames(sp_sce), collapse = ", "), "\n")

EC.counts <- assay(sp_sce, "counts")
#! 注：这里sp_sce是SCE对象，不是Seurat对象。但即使不是Seurat对象也能正常获取EC.counts和EC.metadata
EC.metadata <- as.data.frame(colData(sp_sce))
EC.gene <- rownames(EC.counts)
cat("counts  :", paste(dim(EC.counts), collapse = " x "), "(genes x cells)\n")
cat("label col:", label_col, "\n")


[02:06:52] 1. read h5ad with counts as X and no layers


SCE dim: 6171 x 95054 
assays: counts 
counts  : 6171 x 95054 (genes x cells)
label col: afca_annotation 


In [42]:
# ---- 2. pseudo_group (reference step 1) -----------------------------------
step("2. add pseudo_group (10 cells per group; leftovers -> group 0)")
EC.metadata <- EC.metadata[label_col]
colnames(EC.metadata) <- c("cell_type")
EC.metadata <- mutate_all(EC.metadata, as.character)
cell_type <- unique(EC.metadata$cell_type)
EC.metadata.list <- lapply(X = cell_type, FUN = function(x) {
    data.use <- EC.metadata %>% filter(cell_type == x)
    cnt <- nrow(data.use)
    if (cnt <= 10) {
        data.use$pseudo_group <- rep(1, cnt)
    } else {
        data.use$pseudo_group <- sample(c(rep(seq(1, floor(cnt / 10)), 10), rep(0, cnt %% 10)))
    }
    data.use
})
EC.metadata <- do.call(what = "rbind", args = EC.metadata.list)
cell.keep <- rownames(EC.metadata)
cat("cell types:", length(cell_type), "\n")
cat("cells kept:", length(cell.keep), "\n")
cat("pseudo groups:", length(unique(EC.metadata$pseudo_group)), "\n")


[02:07:05] 2. add pseudo_group (10 cells per group; leftovers -> group 0)


cell types: 76 
cells kept: 95054 
pseudo groups: 473 


In [43]:
# ---- 3. ortholog gene (reference step 2) ----------------------------------
# B2A is intentionally skipped: the two input h5ad files already contain only
# the 8910 one-to-one orthologs. gene.keep is therefore all genes.
step("3. gene.keep = all genes (B2A/A/B mapping skipped by design)")
gene.keep <- EC.gene
cat("genes kept:", length(gene.keep), "\n")


[02:07:06] 3. gene.keep = all genes (B2A/A/B mapping skipped by design)
genes kept: 6171 


In [44]:
# ---- 4. reorder cells, build SCE (reference step 3) -----------------------
step("4. reorder cells and build SCE")
EC.counts <- EC.counts[, rownames(EC.metadata)]
sce <- SingleCellExperiment(assays = list(counts = EC.counts),
                            colData = EC.metadata)
cd <- as.data.frame(colData(sce))
cd[] <- lapply(cd, as.factor)
rownames(cd) <- colnames(sce)
colData(sce) <- DataFrame(cd)
sce <- sce[gene.keep, cell.keep]
cat("SCE dim:", paste(dim(sce), collapse = " x "), "\n")


[02:07:06] 4. reorder cells and build SCE


SCE dim: 6171 x 95054 


In [45]:
# ---- 5. pseudo-cell counts (reference step 4.1) ---------------------------
step("5. aggregate UMI counts into pseudo-cells")
groups <- colData(sce)[, c("cell_type", "pseudo_group")]
pb <- aggregate.Matrix(t(counts(sce)), groupings = groups, fun = "sum")
cat("pb dim:", paste(dim(pb), collapse = " x "), "(pseudo-cells x genes)\n")


[02:07:09] 5. aggregate UMI counts into pseudo-cells


pb dim: 9542 x 6171 (pseudo-cells x genes)


In [46]:
# ---- 6. CP10K + z-score (reference steps 4.2 / 4.3) -----------------------
step("6. CP10K then z-score")
pb.CP10K <- apply(X = pb, MARGIN = 1, FUN = function(x) { x * 1000 / sum(x) })
pb.norm  <- apply(X = pb.CP10K, MARGIN = 1, FUN = function(x) {
    if (sd(x) != 0) (x - mean(x)) / sd(x) else x - mean(x)
})
cat("pb.CP10K dim :", paste(dim(pb.CP10K), collapse = " x "), "(genes x pseudo-cells)\n")
cat("pb.norm dim  :", paste(dim(pb.norm), collapse = " x "), "(pseudo-cells x genes)\n")


[02:07:14] 6. CP10K then z-score


pb.CP10K dim : 6171 x 9542 (genes x pseudo-cells)
pb.norm dim  : 9542 x 6171 (pseudo-cells x genes)


In [47]:
# ---- 7. pseudo-cell metadata (reference step 5) ---------------------------
step("7. build pseudo-cell metadata")
pb.metadata <- colData(sce) %>%
    as.data.frame() %>%
    mutate_all(as.character) %>%
    unique() %>%
    `row.names<-`(NULL)
pb.metadata <- pb.metadata %>%
    mutate(pseudo_cell = str_c(cell_type, pseudo_group, sep = "_")) %>%
    column_to_rownames(var = "pseudo_cell") %>%
    select(cell_type)
cat("pb metadata:", paste(dim(pb.metadata), collapse = " x "), "\n")


[02:07:31] 7. build pseudo-cell metadata
pb metadata: 9542 x 1 


In [48]:
# ---- 8. save pseudo-cell SCE (reference step 6) ---------------------------
step("8. save pb.RDS")
pb <- pb[rownames(pb.metadata), ]
pb.sce <- SingleCellExperiment(assays = list(counts = t(pb)),
                            colData = pb.metadata)
saveRDS(pb.sce, file.path(RDS_out_path))


[02:07:32] 8. save pb.RDS


In [49]:
# ---- 9. marker calculation (reference step 7) -----------------------------
step("9. FindAllMarkers on z-scored pseudo-cell matrix")
# Seurat replaces '_' in feature names with '-'; keep a map to restore them.
orig_genes <- colnames(pb.norm)
safe_genes <- gsub("_", "-", orig_genes)
EC <- CreateSeuratObject(counts = t(pb.norm), assay = "RNA")
EC@meta.data$orig.ident <- pb.metadata$cell_type[match(rownames(EC@meta.data),
                                                    rownames(pb.metadata))]
ident <- EC@meta.data[, "orig.ident"]
names(ident) <- rownames(EC@meta.data)
EC@active.ident <- factor(ident)

EC.markers <- FindAllMarkers(EC, test.use = "wilcox", only.pos = TRUE,
                                min.pct = 0.2, logfc.threshold = 1.25)
markers <- select(EC.markers, cluster, gene, p_val_adj, p_val, avg_log2FC) %>%
    filter(p_val_adj < 0.05)
# restore the original gene names (undo Seurat's '_' -> '-' replacement)
markers$gene <- orig_genes[match(markers$gene, safe_genes)]
write.table(markers, file.path(marker_out_path),
            quote = FALSE, sep = "\t", row.names = FALSE)
cat("markers:", nrow(markers), "rows\n")
step("done")



[02:07:47] 9. FindAllMarkers on z-scored pseudo-cell matrix


Calculating cluster Kenyon cell
Calculating cluster Poxn neuron


  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=04s  


Calculating cluster T neuron T2


  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=02s  


...

  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=03s  


Calculating cluster transmedullary neuron Tm5c


  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=07s  


Calculating cluster transmedullary neuron Tm9


  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=06s  
markers: 4419 rows

[02:26:17] done
